# 5G HetNet Optimisation — PSO-KM(CM2)
### B.Tech Thesis Demo Notebook
**Thesis:** Optimal User Association, Backhaul Routing and Switching Off in 5G HetNets with Mesh mm-Wave Backhaul Links  
**Institution:** CVR College of Engineering (JNTUH) | 2023–24  
**Contributors:** E. Abhilash, **B. Neela Reddy**, K. Vijay Varun Tej  
**Supervisor:** Mr. L. Manjunath, Associate Professor  

---

This notebook reconstructs the thesis simulation in Python and demonstrates:
1. **Network initialisation** — 200 nodes in a 200×200 m area
2. **PSO-KM(CM2) clustering** — joint optimisation with enhanced cluster matching
3. **Convergence: Fixed vs Adaptive inertia** — a new CM2+ extension
4. **HetNet performance** — PSO vs OEERP, LEACH, DRINA, BCDCA
5. **Confidence intervals** — statistical validation across 10 random seeds


In [ ]:
# ── Imports & Setup ─────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from pso_km_cm2 import PSOKM_CM2
from hetnet_simulation import HetNetSimulation, simulate_protocol

plt.style.use('dark_background')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True,
                     'grid.alpha': 0.3, 'font.size': 10})

print("All modules loaded. Python reconstruction of MATLAB/Simulink thesis code.")


## 1. Network Initialisation
200 nodes distributed randomly in a 200×200 m area.  
The macrocell base station sits at the centre (100, 100).


In [ ]:
# Seed for reproducibility
RNG = np.random.default_rng(42)
coords = RNG.uniform(0, 200, size=(200, 2))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(coords[:, 0], coords[:, 1], s=15, c='#888', alpha=0.7, label='Small Cell Nodes')
ax.scatter(100, 100, s=250, c='red', marker='D', zorder=5, label='Macro BS (100,100)')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Fig 6.1 — Node Initialisation: 200 Nodes in 200×200 m Area', fontweight='bold')
ax.legend(); ax.set_xlim(0,200); ax.set_ylim(0,200)
plt.tight_layout(); plt.show()
print(f"Nodes initialised: {len(coords)}")


## 2. PSO-KM(CM2) Clustering
Running PSO-KM with **CM2 Enhanced Cluster Matching** (the thesis novel contribution).

**CM2 key idea:** before each PSO velocity update, rearrange each particle's centroids  
to best match the global best particle ordering. This prevents centroid-ordering  
divergence that plagues standard PSO-KM.

**Parameters (thesis Table 1):** K=10, swarm=6, max_iter=40, w=0.7, c1=0.2, c2=0.2


In [ ]:
K = 10
pso = PSOKM_CM2(K=K, swarm_size=6, max_iter=40, w=0.7, c1=0.2, c2=0.2, seed=42)
centroids, final_mse = pso.fit(coords)
assignments = pso.get_assignments(coords, centroids)

COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
          '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf']

# Select cluster heads (node closest to centroid)
heads = []
for k in range(K):
    members = np.where(assignments == k)[0]
    if len(members) == 0: continue
    dists = np.linalg.norm(coords[members] - centroids[k], axis=1)
    heads.append(members[np.argmin(dists)])

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: random initialisation
axes[0].scatter(coords[:,0], coords[:,1], s=12, c='#888', alpha=0.7)
axes[0].scatter(100,100,s=200,c='red',marker='D',label='Base Station',zorder=5)
axes[0].set_title('Fig 6.1 — Before Clustering'); axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')
axes[0].legend()

# Right: clustered
for k in range(K):
    mask = assignments == k
    axes[1].scatter(coords[mask,0], coords[mask,1], s=12, c=COLORS[k], alpha=0.6)
head_coords = coords[heads]
axes[1].scatter(head_coords[:,0], head_coords[:,1], s=140,
                c=[COLORS[assignments[h]] for h in heads],
                marker='*', edgecolors='white', lw=0.5, zorder=6, label='Cluster Heads')
for hc in head_coords:
    axes[1].plot([hc[0],100],[hc[1],100], color='white', lw=0.4, alpha=0.3)
axes[1].scatter(100,100,s=200,c='red',marker='D',zorder=7,label='Macro BS')
axes[1].set_title(f'Fig 6.2 — PSO-KM(CM2) | Final MSE: {final_mse:.2f}')
axes[1].set_xlabel('X (m)'); axes[1].set_ylabel('Y (m)'); axes[1].legend()

plt.tight_layout(); plt.show()
print(f"Cluster heads selected: {len(heads)} | Final MSE: {final_mse:.4f}")


## 3. Convergence: Fixed vs Adaptive Inertia
**New CM2+ extension:** Instead of fixed w=0.7, we linearly decrease inertia from 0.9→0.4:

```
w(t) = w_start - (w_start - w_end) × (t / max_iter)
```

Higher initial w = more exploration early on.  
Lower final w = finer exploitation near convergence.  
This typically produces lower final MSE for the same iteration budget.


In [ ]:
# Fixed inertia (thesis default)
pso_fixed = PSOKM_CM2(K=10, swarm_size=6, max_iter=40, w=0.7, c1=0.2, c2=0.2, seed=42)
pso_fixed.fit(coords)

# Adaptive inertia (CM2+ extension)
pso_adapt = PSOKM_CM2(K=10, swarm_size=6, max_iter=40,
                       adaptive_inertia=True, w_start=0.9, w_end=0.4,
                       c1=0.2, c2=0.2, seed=42)
pso_adapt.fit(coords)

fig, ax = plt.subplots(figsize=(10, 4.5))
iters = range(len(pso_fixed.mse_history))
ax.plot(iters, pso_fixed.mse_history, color='#ff7f0e', lw=1.8, linestyle='--',
        label=f'Fixed w=0.7 (final MSE: {pso_fixed.mse_history[-1]:.2f})')
ax.plot(range(len(pso_adapt.mse_history)), pso_adapt.mse_history, color='#1f77b4', lw=2.2,
        label=f'Adaptive w: 0.9→0.4 (final MSE: {pso_adapt.mse_history[-1]:.2f})')
ax.fill_between(range(len(pso_adapt.mse_history)), pso_adapt.mse_history, alpha=0.12, color='#1f77b4')

# Mark convergence
for hist, label, color in [(pso_fixed.mse_history, 'fixed', '#ff7f0e'),
                             (pso_adapt.mse_history, 'adaptive', '#1f77b4')]:
    for i in range(1, len(hist)):
        if hist[i-1] > 0 and abs(hist[i]-hist[i-1])/hist[i-1] < 0.01:
            ax.axvline(i, color=color, lw=0.8, linestyle=':', alpha=0.7)
            break

ax.set_xlabel('Iteration'); ax.set_ylabel('MSE (clustering fitness)')
ax.set_title('Fig 6.4 — PSO Convergence: Fixed vs Adaptive Inertia Weight', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

impr = (1 - pso_adapt.mse_history[-1]/pso_fixed.mse_history[-1]) * 100
print(f"Adaptive inertia improvement: {impr:+.1f}% lower final MSE")


## 4. HetNet Simulation — PSO vs Baseline Protocols
Running 6 rounds (each = 50 ms) and comparing PSO-KM(CM2) against:
- **OEERP** — Energy-Efficient Routing Protocol
- **LEACH** — Low Energy Adaptive Clustering Hierarchy
- **DRINA** — Data Routing for In-Network Aggregation
- **BCDCA** — Balanced Cluster-based Data Collection Algorithm


In [ ]:
# PSO simulation
sim = HetNetSimulation(seed=42)
pso_results = sim.run(rounds=6)

# Baselines
baselines = {p: simulate_protocol(p, rounds=6, seed=42)
             for p in ['OEERP', 'LEACH', 'DRINA', 'BCDCA']}
all_results = {'PSO': pso_results, **baselines}

TIME_LABELS = [50, 100, 150, 200, 250, 300]
PROTO_COLORS = {'PSO':'#1f77b4','OEERP':'#ff7f0e','LEACH':'#f0c030','DRINA':'#9467bd','BCDCA':'#2ca02c'}

metrics = [
    ('network_lifetime_ms', 'Network Lifetime (ms)', 'Fig 6.5'),
    ('pdr',                 'Packet Delivery Ratio (%)', 'Fig 6.6'),
    ('throughput',          'Throughput (bps)', 'Fig 6.7'),
    ('energy_joules',       'Energy Consumption (J)', 'Fig 6.8'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
x = np.arange(6)
n_p = len(all_results)
offsets = np.linspace(-(n_p-1)/2, (n_p-1)/2, n_p) * 0.14

for ax_i, (key, ylabel, fig_lbl) in enumerate(metrics):
    ax = axes[ax_i]
    for i, (proto, results) in enumerate(all_results.items()):
        vals = [r[key] for r in results]
        ax.bar(x + offsets[i], vals, 0.14, label=proto,
               color=PROTO_COLORS[proto], edgecolor='black', linewidth=0.3, alpha=0.9)
    ax.set_title(f'{fig_lbl} — {ylabel}', fontweight='bold')
    ax.set_xlabel('Time (ms)'); ax.set_ylabel(ylabel)
    ax.set_xticks(x); ax.set_xticklabels(TIME_LABELS)
    ax.legend(fontsize=8, ncol=3)

plt.tight_layout(); plt.show()

print("\nPSO results at each time slot:")
print(f"{'Time':>8} {'Lifetime':>12} {'PDR%':>8} {'Throughput':>14} {'Energy(J)':>10}")
for r in pso_results:
    print(f"{r['time_ms']:>8} {r['network_lifetime_ms']:>12,} {r['pdr']:>8.1f} {r['throughput']:>14,.0f} {r['energy_joules']:>10.1f}")


## 5. Statistical Validation — Confidence Intervals (10 Seeds)
A single simulation run can be lucky or unlucky depending on node placement.  
Running with 10 different random seeds gives mean ± std — a statistically sound result.

> **This addresses one of the main weaknesses of the original thesis**: single-run results  
> (especially PDR=100%) that look suspiciously clean. The confidence bands show the true  
> variability and make PSO's advantage more credible, not less.


In [ ]:
N_SEEDS = 10
print(f"Running {N_SEEDS} seeds × 5 protocols × 6 rounds...")

ci_data = {p: [] for p in all_results}
for s in range(N_SEEDS):
    sim = HetNetSimulation(seed=s * 7 + 1)
    ci_data['PSO'].append(sim.run(rounds=6))
    for p in ['OEERP', 'LEACH', 'DRINA', 'BCDCA']:
        ci_data[p].append(simulate_protocol(p, rounds=6, seed=s*7+1))

# Compute mean ± std
ci_stats = {}
for proto, runs in ci_data.items():
    ci_stats[proto] = {}
    for key, _, _ in metrics:
        vals = np.array([[r[key] for r in run] for run in runs])
        ci_stats[proto][key] = {'mean': vals.mean(axis=0), 'std': vals.std(axis=0)}

print("Done.")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
x_ci = np.arange(6)

for ax_i, (key, ylabel, fig_lbl) in enumerate(metrics):
    ax = axes[ax_i]
    for proto, color in PROTO_COLORS.items():
        mean = ci_stats[proto][key]['mean']
        std  = ci_stats[proto][key]['std']
        ax.plot(x_ci, mean, color=color, lw=2.2, label=proto, marker='o', markersize=4)
        ax.fill_between(x_ci, mean-std, mean+std, color=color, alpha=0.15)
    ax.set_title(f'{ylabel} — Mean ± 1 Std ({N_SEEDS} seeds)', fontweight='bold')
    ax.set_xlabel('Time (ms)'); ax.set_ylabel(ylabel)
    ax.set_xticks(x_ci); ax.set_xticklabels(TIME_LABELS)
    ax.legend(fontsize=8, ncol=3)

plt.tight_layout(); plt.show()


## 6. Summary

| Metric (at 300ms) | PSO-KM(CM2) | Best Baseline |
|---|---|---|
| Network Lifetime (ms) | **highest** | DRINA |
| Packet Delivery Ratio (%) | **~100** | DRINA (~66%) |
| Throughput | **~7.8×10⁴ bps** | DRINA |
| Energy Consumption (J) | moderate | DRINA (lowest) |

### Key contributions
1. **CM2 cluster matching** resolves centroid-ordering divergence in standard PSO-KM  
2. **Adaptive inertia (CM2+)** — new extension, reduces final MSE vs fixed w  
3. **Statistical validation** — 10-seed confidence intervals confirm PSO's advantage is consistent  

### GitHub
Code: [github.com/neelareddybollu/5g-hetnet-pso-km-cm2](https://github.com/neelareddybollu/5g-hetnet-pso-km-cm2)  
Live app: [5g-hetnet-pso-km-cm2.streamlit.app](https://5g-hetnet-pso-km-cm2-e3jgfkedpldcqvxxsmqpno.streamlit.app)
